# Jewelry OpenAI Pricer

Price jewelry auction lots with GPT-4.1 Mini. Input paths are defined explicitly below so the loaded files are visible before running API calls.


## Configuration


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "jewelry_prompt_builder.py").exists():
    JEWELRY_DIR = NOTEBOOK_DIR
else:
    JEWELRY_DIR = Path("Base_OpenAI/jewelry").resolve()

BASE_OPENAI = JEWELRY_DIR.parent
WORKSPACE_ROOT = BASE_OPENAI.parent
sys.path.insert(0, str(BASE_OPENAI))
sys.path.insert(0, str(JEWELRY_DIR))

OUTPUT_DIR = JEWELRY_DIR / "outputs"
JEWELRY_DATA_DIR = WORKSPACE_ROOT / "Data" / "sampled_sets_jewelry"
QUARTERS = ("2025Q1", "2025Q2", "2025Q3", "2025Q4")
QUARTER_FILES = {quarter: JEWELRY_DATA_DIR / f"Jewelry_2025_high_midpoint_error_q{quarter}_sample1000.csv" for quarter in QUARTERS}

MODEL = "gpt-4.1-mini"
INCLUDE_AUCTION_ESTIMATES = False
INCLUDE_MARKET_INDICATORS = True
VARIANT = "with_estimates" if INCLUDE_AUCTION_ESTIMATES else "no_estimates"
TARGET_COL = "ah_price_usd"
ROWS_PER_QUARTER = None  # Set to None for a full run.
MAX_ROWS = None
RANDOM_STATE = 42
RESUME = True
CHECKPOINT_EVERY = 10
MAX_WORKERS = 50
TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 300
MAX_RETRIES = 3
ENV_FILE = BASE_OPENAI / "env"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project dir: { JEWELRY_DIR }")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Data dir:    {JEWELRY_DATA_DIR}")
for quarter, path in QUARTER_FILES.items():
    print(f"{quarter}: {path}")


Project dir: /srv/notebooks/[6.0] ML Pricers/Base_OpenAI/jewelry
Output dir:  /srv/notebooks/[6.0] ML Pricers/Base_OpenAI/jewelry/outputs
Data dir:    /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry
2025Q1: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q1_sample1000.csv
2025Q2: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q2_sample1000.csv
2025Q3: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q3_sample1000.csv
2025Q4: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q4_sample1000.csv


## Imports


In [2]:
import pandas as pd

from openai_pricer import load_explicit_quarter_files, prepare_pricing_frame
from openai_pricer.runner import run_asset_pricer
from jewelry_prompt_builder import build_prompt, build_system_prompt


## Load Quarter Data


In [3]:
raw_df = load_explicit_quarter_files(QUARTER_FILES, verbose=True)
pricing_df = prepare_pricing_frame(
    raw_df,
    rows_per_quarter=ROWS_PER_QUARTER,
    max_rows=MAX_ROWS,
    random_state=RANDOM_STATE,
    dedupe_key_col="lot_url",
    verbose=True,
)
print("Rows selected for pricing:", len(pricing_df))
print(pricing_df["source_quarter"].value_counts().sort_index())
pricing_df[[c for c in ['ID', 'source_quarter', 'sale_date', 'product_name', 'primary_metal_specification', 'jewelry_category_and_subtypes', 'ah_low_estimate_usd', 'ah_high_estimate_usd', 'ah_price_usd'] if c in pricing_df.columns]].head()


[openai data] loaded 2025Q1 Jewelry_2025_high_midpoint_error_q2025Q1_sample1000.csv: 1,000 rows
[openai data] loaded 2025Q2 Jewelry_2025_high_midpoint_error_q2025Q2_sample1000.csv: 1,000 rows
[openai data] loaded 2025Q3 Jewelry_2025_high_midpoint_error_q2025Q3_sample1000.csv: 1,000 rows
[openai data] loaded 2025Q4 Jewelry_2025_high_midpoint_error_q2025Q4_sample1000.csv: 1,000 rows
[openai data] combined quarter rows: 4,000
[openai data] quarters: {'2025Q1': 1000, '2025Q2': 1000, '2025Q3': 1000, '2025Q4': 1000}
[openai data] deduplicated lot_url: 4,000->4,000
Rows selected for pricing: 4000
source_quarter
2025Q1    1000
2025Q2    1000
2025Q3    1000
2025Q4    1000
Name: count, dtype: int64


,ID,source_quarter,sale_date,product_name,primary_metal_specification,jewelry_category_and_subtypes,ah_low_estimate_usd,ah_high_estimate_usd,ah_price_usd
0,9ef25f19-3f45-40bb-9e41-e45b66bda0a5,2025Q1,2025-03-25,Paste and Gem-Set Ring,"{'metal_type': 'N/A', 'purity': 'N/A', 'color'...",Ring,506.27,759.40,53.79
1,1390d31c-2d86-471b-93be-126abb14c876,2025Q1,2025-03-04,18K Gold and Lapis Lazuli Beaded Necklace,"{'metal_type': 'Gold', 'purity': '18k', 'color...",Necklace,3000.00,4000.00,490.00
2,d4328a38-4e26-42d3-9e9a-e465c1aeab95,2025Q1,2025-03-04,Group of Quartzite,"{'metal_type': 'Silver-toned metal', 'purity':...",Necklace,400.00,600.00,74.17
3,f977620c-0487-498b-83de-bf56932d7115,2025Q1,2025-03-25,Diamond and Gem-Set Fox Brooch,"{'metal_type': 'N/A', 'purity': 'N/A', 'color'...",Brooch,1012.54,1518.81,188.80
4,6da0f2f4-7021-4e92-a0fc-fe56337011bb,2025Q1,2025-03-04,"Group of Gold, Silver, Jade, Nephrite and Cult...","{'metal_type': 'Gold, Silver', 'purity': '14k,...",Jewelry Set,3000.00,4000.00,800.00


## Preview Prompt

The prompt excludes realized sale prices and existing model prediction columns.


In [4]:
system_prompt = build_system_prompt(INCLUDE_AUCTION_ESTIMATES)
example_prompt = build_prompt(
    pricing_df.iloc[0].to_dict(),
    INCLUDE_AUCTION_ESTIMATES,
    INCLUDE_MARKET_INDICATORS,
)
print("SYSTEM PROMPT:")
print(system_prompt[:2000])
print("\nUSER PROMPT:")
print(example_prompt[:4000])


SYSTEM PROMPT:
You are an expert jewelry appraiser. Price the jewelry lot's expected sale value in USD under normal auction-market conditions. Use only the jewelry and auction metadata provided in the user prompt. Do not use realized sale price, buyer-premium price, or other model prediction columns as inputs. Do not use auction-house estimate fields for valuation even if they appear in metadata. Consider maker, category, metal, weight, ring or chain measurements, gemstone and diamond specifications, pearl details, signatures, certification, origin, sale venue, location, sale date, and available market indicators. Return a realistic nonzero USD valuation. Respond only as JSON matching the requested schema.

USER PROMPT:
Price this jewelry lot using the available fields below. Do not use any realized sale price or model prediction as an input.

Jewelry data:
Asset Type: Jewelry
Product Name: Paste and Gem-Set Ring
Brand And Model Identifiers: Brand: Zoltan White & Co.; Collection: N/A; 

## Run OpenAI Pricing

Requires `OPENAI_API_KEY` in `Base_OpenAI/env` or the environment. Keep `ROWS_PER_QUARTER=10` for a smoke test; set it to `None` for a full run.


In [5]:
run_asset_pricer(
    asset_name="jewelry",
    quarter_files=QUARTER_FILES,
    output_dir=OUTPUT_DIR,
    output_prefix=f"openai_gpt41mini_jewelry_{VARIANT}",
    model=MODEL,
    model_label=f"OpenAI GPT-4.1 Mini Jewelry - {'With Estimates' if INCLUDE_AUCTION_ESTIMATES else 'No Estimates'}",
    build_prompt=build_prompt,
    build_system_prompt=build_system_prompt,
    actual_col=TARGET_COL,
    include_auction_estimates=INCLUDE_AUCTION_ESTIMATES,
    include_market_indicators=INCLUDE_MARKET_INDICATORS,
    max_rows=MAX_ROWS,
    rows_per_quarter=ROWS_PER_QUARTER,
    random_state=RANDOM_STATE,
    resume=RESUME,
    checkpoint_every=CHECKPOINT_EVERY,
    max_workers=MAX_WORKERS,
    temperature=TEMPERATURE,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    max_retries=MAX_RETRIES,
    env_file=ENV_FILE,
)


[openai jewelry] output_prefix=openai_gpt41mini_jewelry_no_estimates, model=gpt-4.1-mini, include_auction_estimates=False, include_market_indicators=True, workers=50
[openai jewelry] predictions=/srv/notebooks/[6.0] ML Pricers/Base_OpenAI/jewelry/outputs/openai_gpt41mini_jewelry_no_estimates_predictions.csv
[openai data] loaded 2025Q1 Jewelry_2025_high_midpoint_error_q2025Q1_sample1000.csv: 1,000 rows
[openai data] loaded 2025Q2 Jewelry_2025_high_midpoint_error_q2025Q2_sample1000.csv: 1,000 rows
[openai data] loaded 2025Q3 Jewelry_2025_high_midpoint_error_q2025Q3_sample1000.csv: 1,000 rows
[openai data] loaded 2025Q4 Jewelry_2025_high_midpoint_error_q2025Q4_sample1000.csv: 1,000 rows
[openai data] combined quarter rows: 4,000
[openai data] quarters: {'2025Q1': 1000, '2025Q2': 1000, '2025Q3': 1000, '2025Q4': 1000}
[openai data] deduplicated lot_url: 4,000->4,000
[openai jewelry] rows selected: 4,000
source_quarter
2025Q1    1000
2025Q2    1000
2025Q3    1000
2025Q4    1000
[openai jewel